Cell 1 — Imports and configuration

In [1]:
import json
import random
import shutil
import sys
import time
from pathlib import Path
import os

import numpy as np
import torch
from datasets import DatasetDict, load_dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    pipeline,
    set_seed,
)

# Model and output paths
BASE_MODEL = "distilbert-base-uncased"

FINAL_MODEL_DIR = Path(
    "../saved_models/distilbert_fintech_pt"
)

CHECKPOINT_DIR = Path(
    "../saved_models/distilbert_checkpoints"
)

METRICS_DIR = Path(
    "../saved_models/training_metrics"
)

# Training settings
MAX_LENGTH = 128
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
LEARNING_RATE = 2e-5
RANDOM_SEED = 42

# Reproducibility
set_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if hasattr(torch, "xpu"):
    print("Intel XPU available:", torch.xpu.is_available())
else:
    print("Intel XPU support: unavailable")

print("Training will be forced to CPU.")
print("Current working directory:", Path.cwd())

Python executable: c:\Users\Wang Song\Downloads\fintech-triage-agent\backend\venv\Scripts\python.exe
Python version: 3.13.6 (tags/v3.13.6:4e66535, Aug  6 2025, 14:36:00) [MSC v.1944 64 bit (AMD64)]
PyTorch version: 2.13.0+xpu
CUDA available: False
Intel XPU available: True
Training will be forced to CPU.
Current working directory: c:\Users\Wang Song\Downloads\fintech-triage-agent\backend\notebooks


Cell 2 — Prepare temporary output directories
- deletes old checkpoints and metrics
- It does not delete your current final model.

In [2]:
# Remove previous temporary checkpoints and metrics.
# Do not delete the existing final model yet.

for directory in [CHECKPOINT_DIR, METRICS_DIR]:
    if directory.exists():
        print(f"Removing old temporary directory: {directory}")
        shutil.rmtree(directory)

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print("Temporary training directories are ready.")

if FINAL_MODEL_DIR.exists():
    print()
    print("Existing trained model found:")
    print(FINAL_MODEL_DIR.resolve())
    print(
        "It will remain untouched until the new model "
        "finishes training and evaluation successfully."
    )
else:
    print()
    print("No existing final model was found.")

Temporary training directories are ready.

Existing trained model found:
C:\Users\Wang Song\Downloads\fintech-triage-agent\backend\saved_models\distilbert_fintech_pt
It will remain untouched until the new model finishes training and evaluation successfully.


Cell 3 — Load Banking77 and create validation split

In [3]:
raw_dataset = load_dataset("mteb/banking77")

print("Original dataset:")
print(raw_dataset)
print()

# Create a stratified 90/10 training-validation split.
# The official test set remains untouched.
all_indices = np.arange(len(raw_dataset["train"]))
all_labels = np.array(raw_dataset["train"]["label"])

train_indices, validation_indices = train_test_split(
    all_indices,
    test_size=0.10,
    random_state=RANDOM_SEED,
    stratify=all_labels,
)

dataset = DatasetDict(
    {
        "train": raw_dataset["train"].select(
            train_indices.tolist()
        ),
        "validation": raw_dataset["train"].select(
            validation_indices.tolist()
        ),
        "test": raw_dataset["test"],
    }
)

print("Prepared dataset:")
print(dataset)
print()

print("Training examples:", len(dataset["train"]))
print("Validation examples:", len(dataset["validation"]))
print("Reserved test examples:", len(dataset["test"]))
print()

print("Dataset columns:", dataset["train"].column_names)
print()
print("Example training row:")
print(dataset["train"][0])

Original dataset:
DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 9993
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 3076
    })
})

Prepared dataset:
DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 8993
    })
    validation: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 3076
    })
})

Training examples: 8993
Validation examples: 1000
Reserved test examples: 3076

Dataset columns: ['text', 'label', 'label_text']

Example training row:
{'text': 'I am still waiting on my card?', 'label': 11, 'label_text': 'card_arrival'}


Cell 4 — Create readable label mappings

In [4]:
label_pairs = {
    int(label_id): label_text
    for label_id, label_text in zip(
        raw_dataset["train"]["label"],
        raw_dataset["train"]["label_text"],
    )
}

id2label = dict(sorted(label_pairs.items()))

label2id = {
    label_name: label_id
    for label_id, label_name in id2label.items()
}

NUM_LABELS = len(id2label)

assert NUM_LABELS == 77, (
    f"Expected 77 labels, but found {NUM_LABELS}"
)

assert len(label2id) == 77

print("Number of Banking77 labels:", NUM_LABELS)
print()
print("First 15 label mappings:")

for label_id in range(15):
    print(f"{label_id:02d}: {id2label[label_id]}")

print()
print(
    "lost_or_stolen_card ID:",
    label2id["lost_or_stolen_card"],
)

Number of Banking77 labels: 77

First 15 label mappings:
00: activate_my_card
01: age_limit
02: apple_pay_or_google_pay
03: atm_support
04: automatic_top_up
05: balance_not_updated_after_bank_transfer
06: balance_not_updated_after_cheque_or_cash_deposit
07: beneficiary_not_allowed
08: cancel_transfer
09: card_about_to_expire
10: card_acceptance
11: card_arrival
12: card_delivery_estimate
13: card_linking
14: card_not_working

lost_or_stolen_card ID: 41


Cell 5 — Load tokenizer and tokenize all splits

In [5]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize_batch(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_datasets = dataset.map(
    tokenize_batch,
    batched=True,
    desc="Tokenizing Banking77",
)

# Dynamic padding pads only to the longest sequence
# in each batch instead of always padding to 128.
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt",
)

print(tokenized_datasets)
print()

print(
    "Tokenized training columns:",
    tokenized_datasets["train"].column_names,
)

Tokenizing Banking77:   0%|          | 0/8993 [00:00<?, ? examples/s]

Tokenizing Banking77:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing Banking77:   0%|          | 0/3076 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8993
    })
    validation: Dataset({
        features: ['text', 'label', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['text', 'label', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3076
    })
})

Tokenized training columns: ['text', 'label', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask']


Cell 6 — Initialize a fresh DistilBERT model

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

print("Fresh base model loaded:", BASE_MODEL)
print("Number of output labels:", model.config.num_labels)

lost_card_id = model.config.label2id[
    "lost_or_stolen_card"
]

print(
    "Verified mapping:",
    lost_card_id,
    "->",
    model.config.id2label[lost_card_id],
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fresh base model loaded: distilbert-base-uncased
Number of output labels: 77
Verified mapping: 41 -> lost_or_stolen_card


Cell 7 — Define evaluation metrics

In [7]:
def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(
        labels,
        predictions,
    )

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        )
    )

    weighted_precision, weighted_recall, weighted_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted",
            zero_division=0,
        )
    )

    return {
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_precision,
        "weighted_recall": weighted_recall,
        "weighted_f1": weighted_f1,
    }

print("Evaluation metric function is ready.")

Evaluation metric function is ready.


Cell 8 — Configure training and progress logging

In [8]:
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),

    # Core training settings
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,

    # Evaluate and save after every epoch
    eval_strategy="epoch",
    save_strategy="epoch",

    # Restore the checkpoint with the best validation macro F1
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    # Limit checkpoint disk usage
    save_total_limit=2,

    # Plain-text progress logging
    logging_strategy="steps",
    logging_steps=50,
    logging_first_step=True,
    disable_tqdm=True,
    report_to="none",
    log_level="info",

    # Reproducibility
    seed=RANDOM_SEED,
    data_seed=RANDOM_SEED,

    # Force CPU
    use_cpu=True,

    remove_unused_columns=True,
)

print("Training configuration created.")
print("Progress will print every 50 training steps.")
print("Validation will run after every epoch.")
print("Number of epochs:", NUM_EPOCHS)

Training configuration created.
Progress will print every 50 training steps.
Validation will run after every epoch.
Number of epochs: 3


Cell 9 — Create the Trainer

In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2,
            early_stopping_threshold=0.001,
        )
    ],
)

print("Trainer is ready.")
print(
    "Training examples:",
    len(tokenized_datasets["train"]),
)
print(
    "Validation examples:",
    len(tokenized_datasets["validation"]),
)
print(
    "Reserved test examples:",
    len(tokenized_datasets["test"]),
)

Trainer is ready.
Training examples: 8993
Validation examples: 1000
Reserved test examples: 3076


Cell 10 — Train the model

In [10]:
training_start_time = time.time()

print("=" * 60)
print("TRAINING STARTED")
print("=" * 60)
print("Epochs:", NUM_EPOCHS)
print(
    "Training examples:",
    len(tokenized_datasets["train"]),
)
print(
    "Validation examples:",
    len(tokenized_datasets["validation"]),
)
print("Training batch size:", TRAIN_BATCH_SIZE)
print("Progress will print every 50 steps.")
print()

# Fresh training run. Do not use resume_from_checkpoint=True.
train_result = trainer.train()

training_duration_seconds = (
    time.time() - training_start_time
)

training_duration_minutes = (
    training_duration_seconds / 60
)

training_duration_hours = (
    training_duration_minutes / 60
)

print()
print("=" * 60)
print("TRAINING FINISHED")
print("=" * 60)
print(
    f"Duration: {training_duration_minutes:.2f} minutes"
)
print(
    f"Duration: {training_duration_hours:.2f} hours"
)
print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint,
)
print(
    "Best validation macro F1:",
    trainer.state.best_metric,
)
print()
print("Training result:")
print(train_result.metrics)

[transformers] The following columns in the Training set don't have a corresponding argument in `DistilBertForSequenceClassification.forward` and have been ignored: token_type_ids, label_text, text. If token_type_ids, label_text, text are not expected by `DistilBertForSequenceClassification.forward`,  you can safely ignore this message.
[transformers] ***** Running training *****
[transformers]   Num examples = 8,993
[transformers]   Num Epochs = 3
[transformers]   Num update steps per epoch = 1,125
[transformers]   Instantaneous batch size per device = 8
[transformers]   Total train batch size (w. parallel, distributed & accumulation) = 8
[transformers]   Gradient Accumulation steps = 1
[transformers]   Total optimization steps = 3,375
[transformers]   Number of trainable parameters = 67,012,685


TRAINING STARTED
Epochs: 3
Training examples: 8993
Validation examples: 1000
Training batch size: 8
Progress will print every 50 steps.

{'loss': '4.37', 'grad_norm': '2.81', 'learning_rate': '2e-05', 'epoch': '0.0008889'}
{'loss': '4.339', 'grad_norm': '2.734', 'learning_rate': '1.971e-05', 'epoch': '0.04444'}
{'loss': '4.26', 'grad_norm': '4.874', 'learning_rate': '1.941e-05', 'epoch': '0.08889'}
{'loss': '4.116', 'grad_norm': '5.299', 'learning_rate': '1.912e-05', 'epoch': '0.1333'}
{'loss': '3.91', 'grad_norm': '6.041', 'learning_rate': '1.882e-05', 'epoch': '0.1778'}
{'loss': '3.688', 'grad_norm': '5.973', 'learning_rate': '1.852e-05', 'epoch': '0.2222'}
{'loss': '3.534', 'grad_norm': '6.755', 'learning_rate': '1.823e-05', 'epoch': '0.2667'}
{'loss': '3.389', 'grad_norm': '8.516', 'learning_rate': '1.793e-05', 'epoch': '0.3111'}
{'loss': '3.227', 'grad_norm': '7.808', 'learning_rate': '1.764e-05', 'epoch': '0.3556'}
{'loss': '3.083', 'grad_norm': '7.649', 'learning_rate': '1.734e-

[transformers] The following columns in the Evaluation set don't have a corresponding argument in `DistilBertForSequenceClassification.forward` and have been ignored: token_type_ids, label_text, text. If token_type_ids, label_text, text are not expected by `DistilBertForSequenceClassification.forward`,  you can safely ignore this message.
[transformers] 
***** Running Evaluation *****
[transformers]   Num examples = 1000
[transformers]   Batch size = 8
[transformers] Saving model checkpoint to ..\saved_models\distilbert_checkpoints\checkpoint-1125
[transformers] Configuration saved in ..\saved_models\distilbert_checkpoints\checkpoint-1125\config.json


{'eval_loss': '1.426', 'eval_accuracy': '0.758', 'eval_macro_precision': '0.7766', 'eval_macro_recall': '0.7183', 'eval_macro_f1': '0.7009', 'eval_weighted_precision': '0.7892', 'eval_weighted_recall': '0.758', 'eval_weighted_f1': '0.7325', 'eval_runtime': '13.78', 'eval_samples_per_second': '72.56', 'eval_steps_per_second': '9.069', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

[transformers] The following columns in the Evaluation set don't have a corresponding argument in `DistilBertForSequenceClassification.forward` and have been ignored: token_type_ids, label_text, text. If token_type_ids, label_text, text are not expected by `DistilBertForSequenceClassification.forward`,  you can safely ignore this message.
[transformers] 
***** Running Evaluation *****
[transformers]   Num examples = 1000
[transformers]   Batch size = 8
[transformers] Saving model checkpoint to ..\saved_models\distilbert_checkpoints\checkpoint-3375
[transformers] Configuration saved in ..\saved_models\distilbert_checkpoints\checkpoint-3375\config.json


{'eval_loss': '0.5398', 'eval_accuracy': '0.889', 'eval_macro_precision': '0.883', 'eval_macro_recall': '0.8756', 'eval_macro_f1': '0.8742', 'eval_weighted_precision': '0.8915', 'eval_weighted_recall': '0.889', 'eval_weighted_f1': '0.8856', 'eval_runtime': '14.08', 'eval_samples_per_second': '71', 'eval_steps_per_second': '8.876', 'epoch': '3'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Model weights saved in ..\saved_models\distilbert_checkpoints\checkpoint-3375\model.safetensors
[transformers] tokenizer config file saved in ..\saved_models\distilbert_checkpoints\checkpoint-3375\tokenizer_config.json
[transformers] 

Training completed. Do not forget to share your model on huggingface.co/models =)


[transformers] Loading best model from ..\saved_models\distilbert_checkpoints\checkpoint-3375 (score: 0.8742451964824486).


{'train_runtime': '1512', 'train_samples_per_second': '17.84', 'train_steps_per_second': '2.232', 'train_loss': '1.47', 'epoch': '3'}

TRAINING FINISHED
Duration: 25.21 minutes
Duration: 0.42 hours
Best checkpoint: ..\saved_models\distilbert_checkpoints\checkpoint-3375
Best validation macro F1: 0.8742451964824486

Training result:
{'train_runtime': 1512.0637, 'train_samples_per_second': 17.843, 'train_steps_per_second': 2.232, 'train_loss': 1.4696266342445656, 'epoch': 3.0}


Cell 11 — Evaluate on the untouched test set

In [11]:
final_metrics = trainer.evaluate(
    eval_dataset=tokenized_datasets["test"],
    metric_key_prefix="test",
)

print("=" * 65)
print("FINAL RESERVED TEST METRICS")
print("=" * 65)

for metric_name, metric_value in final_metrics.items():
    if isinstance(metric_value, float):
        print(f"{metric_name}: {metric_value:.4f}")
    else:
        print(f"{metric_name}: {metric_value}")

[transformers] The following columns in the Evaluation set don't have a corresponding argument in `DistilBertForSequenceClassification.forward` and have been ignored: token_type_ids, label_text, text. If token_type_ids, label_text, text are not expected by `DistilBertForSequenceClassification.forward`,  you can safely ignore this message.
[transformers] 
***** Running Evaluation *****
[transformers]   Num examples = 3076
[transformers]   Batch size = 8
[transformers] early stopping required metric_for_best_model, but did not find eval_macro_f1 so early stopping is disabled


{'test_loss': '0.5788', 'test_accuracy': '0.8774', 'test_macro_precision': '0.8783', 'test_macro_recall': '0.8774', 'test_macro_f1': '0.8722', 'test_weighted_precision': '0.8784', 'test_weighted_recall': '0.8774', 'test_weighted_f1': '0.8723', 'test_runtime': '26.7', 'test_samples_per_second': '115.2', 'test_steps_per_second': '14.42', 'epoch': '3'}
FINAL RESERVED TEST METRICS
test_loss: 0.5788
test_accuracy: 0.8774
test_macro_precision: 0.8783
test_macro_recall: 0.8774
test_macro_f1: 0.8722
test_weighted_precision: 0.8784
test_weighted_recall: 0.8774
test_weighted_f1: 0.8723
test_runtime: 26.6956
test_samples_per_second: 115.2250
test_steps_per_second: 14.4220
epoch: 3.0000


Cell 12 — Generate detailed class-level results

In [12]:
prediction_output = trainer.predict(
    tokenized_datasets["test"]
)

test_logits = prediction_output.predictions
test_labels = prediction_output.label_ids

test_predictions = np.argmax(
    test_logits,
    axis=-1,
)

target_names = [
    id2label[label_id]
    for label_id in range(NUM_LABELS)
]

report_dict = classification_report(
    test_labels,
    test_predictions,
    labels=list(range(NUM_LABELS)),
    target_names=target_names,
    output_dict=True,
    zero_division=0,
)

print("=" * 60)
print("DETAILED CLASSIFICATION SUMMARY")
print("=" * 60)

print(
    "Accuracy:",
    f"{report_dict['accuracy']:.4f}",
)

print(
    "Macro precision:",
    f"{report_dict['macro avg']['precision']:.4f}",
)

print(
    "Macro recall:",
    f"{report_dict['macro avg']['recall']:.4f}",
)

print(
    "Macro F1:",
    f"{report_dict['macro avg']['f1-score']:.4f}",
)

print(
    "Weighted F1:",
    f"{report_dict['weighted avg']['f1-score']:.4f}",
)

[transformers] The following columns in the test set don't have a corresponding argument in `DistilBertForSequenceClassification.forward` and have been ignored: token_type_ids, label_text, text. If token_type_ids, label_text, text are not expected by `DistilBertForSequenceClassification.forward`,  you can safely ignore this message.
[transformers] 
***** Running Prediction *****
[transformers]   Num examples = 3076
[transformers]   Batch size = 8


DETAILED CLASSIFICATION SUMMARY
Accuracy: 0.8774
Macro precision: 0.8783
Macro recall: 0.8774
Macro F1: 0.8722
Weighted F1: 0.8723


Cell 13 — Display the 15 weakest intents

In [13]:
class_results = []

for label_name in target_names:
    class_metrics = report_dict[label_name]

    class_results.append(
        {
            "intent": label_name,
            "precision": class_metrics["precision"],
            "recall": class_metrics["recall"],
            "f1_score": class_metrics["f1-score"],
            "support": int(class_metrics["support"]),
        }
    )

class_results_sorted = sorted(
    class_results,
    key=lambda item: item["f1_score"],
)

print("=" * 90)
print("15 WEAKEST INTENTS BY F1 SCORE")
print("=" * 90)

print(
    f"{'Intent':40s}"
    f"{'Precision':>11s}"
    f"{'Recall':>11s}"
    f"{'F1':>11s}"
    f"{'Count':>8s}"
)

for row in class_results_sorted[:15]:
    print(
        f"{row['intent'][:40]:40s}"
        f"{row['precision']:11.3f}"
        f"{row['recall']:11.3f}"
        f"{row['f1_score']:11.3f}"
        f"{row['support']:8d}"
    )

15 WEAKEST INTENTS BY F1 SCORE
Intent                                    Precision     Recall         F1   Count
virtual_card_not_working                      0.000      0.000      0.000      40
why_verify_identity                           0.769      0.500      0.606      40
getting_virtual_card                          0.557      0.975      0.709      40
get_disposable_virtual_card                   0.635      0.825      0.717      40
card_swallowed                                1.000      0.575      0.730      40
topping_up_by_card                            0.744      0.725      0.734      40
declined_cash_withdrawal                      0.627      0.925      0.747      40
verify_my_identity                            0.673      0.875      0.761      40
balance_not_updated_after_bank_transfer       0.789      0.750      0.769      40
declined_card_payment                         0.679      0.900      0.774      40
pending_transfer                              0.900      0.692     

Cell 14 — Display the most common classification errors

In [14]:
confusion = confusion_matrix(
    test_labels,
    test_predictions,
    labels=list(range(NUM_LABELS)),
)

confusion_pairs = []

for actual_id in range(NUM_LABELS):
    for predicted_id in range(NUM_LABELS):
        if actual_id == predicted_id:
            continue

        error_count = int(
            confusion[actual_id, predicted_id]
        )

        if error_count > 0:
            confusion_pairs.append(
                {
                    "actual": id2label[actual_id],
                    "predicted": id2label[predicted_id],
                    "count": error_count,
                }
            )

confusion_pairs.sort(
    key=lambda item: item["count"],
    reverse=True,
)

print("=" * 90)
print("20 MOST COMMON CLASSIFICATION ERRORS")
print("=" * 90)

print(
    f"{'Actual intent':36s}"
    f"{'Predicted intent':36s}"
    f"{'Count':>8s}"
)

for item in confusion_pairs[:20]:
    print(
        f"{item['actual'][:36]:36s}"
        f"{item['predicted'][:36]:36s}"
        f"{item['count']:8d}"
    )

20 MOST COMMON CLASSIFICATION ERRORS
Actual intent                       Predicted intent                       Count
virtual_card_not_working            getting_virtual_card                      21
why_verify_identity                 verify_my_identity                        17
virtual_card_not_working            get_disposable_virtual_card               15
card_swallowed                      declined_cash_withdrawal                  13
pending_transfer                    transfer_timing                            7
declined_transfer                   declined_card_payment                      6
transfer_into_account               topping_up_by_card                         6
beneficiary_not_allowed             failed_transfer                            5
card_delivery_estimate              card_arrival                               5
get_disposable_virtual_card         getting_virtual_card                       5
getting_spare_card                  order_physical_card                 

Cell 15 — Replace the old model and save metrics

In [15]:
backup_dir = Path(
    "../saved_models/distilbert_fintech_pt_backup"
)

# Remove an old incomplete backup if one exists.
if backup_dir.exists():
    shutil.rmtree(backup_dir)

# Temporarily move the previous model to a backup location.
if FINAL_MODEL_DIR.exists():
    print("Backing up the previous model...")
    FINAL_MODEL_DIR.rename(backup_dir)

try:
    # Trainer contains the best validation checkpoint because
    # load_best_model_at_end=True.
    trainer.save_model(str(FINAL_MODEL_DIR))
    tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

except Exception:
    # If saving fails, restore the old model.
    if FINAL_MODEL_DIR.exists():
        shutil.rmtree(FINAL_MODEL_DIR)

    if backup_dir.exists():
        backup_dir.rename(FINAL_MODEL_DIR)

    raise

print("New model saved successfully.")

# Delete backup only after the new model is saved.
if backup_dir.exists():
    shutil.rmtree(backup_dir)
    print("Temporary previous-model backup removed.")

# Convert metric values into JSON-safe Python values.
serializable_metrics = {}

for key, value in final_metrics.items():
    if isinstance(value, (np.floating, float)):
        serializable_metrics[key] = float(value)

    elif isinstance(value, (np.integer, int)):
        serializable_metrics[key] = int(value)

    else:
        serializable_metrics[key] = value

# Save overall metrics.
with open(
    METRICS_DIR / "final_metrics.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        serializable_metrics,
        file,
        indent=2,
    )

# Save full classification report.
with open(
    METRICS_DIR / "classification_report.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        report_dict,
        file,
        indent=2,
    )

# Save weak-class results.
with open(
    METRICS_DIR / "weakest_classes.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        class_results_sorted,
        file,
        indent=2,
    )

# Save common confusion pairs.
with open(
    METRICS_DIR / "common_confusions.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        confusion_pairs,
        file,
        indent=2,
    )

print()
print("Final model directory:")
print(FINAL_MODEL_DIR.resolve())

print()
print("Metrics directory:")
print(METRICS_DIR.resolve())

[transformers] Saving model checkpoint to ..\saved_models\distilbert_fintech_pt
[transformers] Configuration saved in ..\saved_models\distilbert_fintech_pt\config.json


Backing up the previous model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Model weights saved in ..\saved_models\distilbert_fintech_pt\model.safetensors
[transformers] tokenizer config file saved in ..\saved_models\distilbert_fintech_pt\tokenizer_config.json
[transformers] tokenizer config file saved in ..\saved_models\distilbert_fintech_pt\tokenizer_config.json


New model saved successfully.
Temporary previous-model backup removed.

Final model directory:
C:\Users\Wang Song\Downloads\fintech-triage-agent\backend\saved_models\distilbert_fintech_pt

Metrics directory:
C:\Users\Wang Song\Downloads\fintech-triage-agent\backend\saved_models\training_metrics


Cell 16 — Verify the saved model locally

In [16]:
saved_model = (
    AutoModelForSequenceClassification.from_pretrained(
        str(FINAL_MODEL_DIR),
        local_files_only=True,
    )
)

saved_tokenizer = AutoTokenizer.from_pretrained(
    str(FINAL_MODEL_DIR),
    local_files_only=True,
)

assert saved_model.config.num_labels == 77

saved_lost_card_id = saved_model.config.label2id[
    "lost_or_stolen_card"
]

assert (
    saved_model.config.id2label[saved_lost_card_id]
    == "lost_or_stolen_card"
)

print("Saved model loaded locally.")
print(
    "Number of labels:",
    saved_model.config.num_labels,
)
print(
    "lost_or_stolen_card ID:",
    saved_lost_card_id,
)
print(
    "Reverse mapping:",
    saved_model.config.id2label[
        saved_lost_card_id
    ],
)
print("Offline model verification passed.")

[transformers] loading configuration file ..\saved_models\distilbert_fintech_pt\config.json
[transformers] Model config DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForSequenceClassification"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_dim": 3072,
  "id2label": {
    "0": "activate_my_card",
    "1": "age_limit",
    "2": "apple_pay_or_google_pay",
    "3": "atm_support",
    "4": "automatic_top_up",
    "5": "balance_not_updated_after_bank_transfer",
    "6": "balance_not_updated_after_cheque_or_cash_deposit",
    "7": "beneficiary_not_allowed",
    "8": "cancel_transfer",
    "9": "card_about_to_expire",
    "10": "card_acceptance",
    "11": "card_arrival",
    "12": "card_delivery_estimate",
    "13": "card_linking",
    "14": "card_not_working",
    "15": "card_payment_fee_charged",
    "16": "card_payment_not_recognised",
    "17": "card_payment_w

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] loading configuration file ..\saved_models\distilbert_fintech_pt\config.json
[transformers] Model config DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForSequenceClassification"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_dim": 3072,
  "id2label": {
    "0": "activate_my_card",
    "1": "age_limit",
    "2": "apple_pay_or_google_pay",
    "3": "atm_support",
    "4": "automatic_top_up",
    "5": "balance_not_updated_after_bank_transfer",
    "6": "balance_not_updated_after_cheque_or_cash_deposit",
    "7": "beneficiary_not_allowed",
    "8": "cancel_transfer",
    "9": "card_about_to_expire",
    "10": "card_acceptance",
    "11": "card_arrival",
    "12": "card_delivery_estimate",
    "13": "card_linking",
    "14": "card_not_working",
    "15": "card_payment_fee_charged",
    "16": "card_payment_not_recognised",
    "17": "card_payment_w

Saved model loaded locally.
Number of labels: 77
lost_or_stolen_card ID: 41
Reverse mapping: lost_or_stolen_card
Offline model verification passed.


Cell 17 — Create the local inference pipeline

In [17]:
classifier = pipeline(
    task="text-classification",
    model=saved_model,
    tokenizer=saved_tokenizer,
    device=-1,
)

print("Local CPU inference pipeline is ready.")

Local CPU inference pipeline is ready.


Cell 18 — Test realistic banking messages

In [18]:
test_messages = [
    "My card was stolen in London.",
    "I lost my bank card while travelling.",
    "My card still has not arrived.",
    "Why was I charged an extra fee abroad?",
    "I do not recognize this cash withdrawal.",
    "I forgot the PIN for my card.",
    "How can I transfer money to another account?",
]

for message in test_messages:
    predictions = classifier(
        message,
        top_k=3,
    )

    print("=" * 80)
    print("Message:", message)

    for rank, prediction in enumerate(
        predictions,
        start=1,
    ):
        print(
            f"{rank}. "
            f"{prediction['label']}: "
            f"{prediction['score']:.4f}"
        )

Message: My card was stolen in London.
1. lost_or_stolen_card: 0.5324
2. compromised_card: 0.0988
3. card_arrival: 0.0701
Message: I lost my bank card while travelling.
1. lost_or_stolen_card: 0.4027
2. card_not_working: 0.1089
3. compromised_card: 0.0562
Message: My card still has not arrived.
1. card_arrival: 0.8249
2. card_delivery_estimate: 0.0447
3. lost_or_stolen_card: 0.0247
Message: Why was I charged an extra fee abroad?
1. transfer_fee_charged: 0.8327
2. top_up_by_bank_transfer_charge: 0.0262
3. exchange_charge: 0.0143
Message: I do not recognize this cash withdrawal.
1. cash_withdrawal_not_recognised: 0.7967
2. declined_cash_withdrawal: 0.0593
3. pending_cash_withdrawal: 0.0348
Message: I forgot the PIN for my card.
1. get_physical_card: 0.5814
2. pin_blocked: 0.2326
3. change_pin: 0.0633
Message: How can I transfer money to another account?
1. transfer_into_account: 0.8057
2. top_up_by_bank_transfer_charge: 0.0308
3. receiving_money: 0.0296


Cell 19 — Assess whether the model is sufficient

In [19]:
accuracy = final_metrics["test_accuracy"]
macro_f1 = final_metrics["test_macro_f1"]
weighted_f1 = final_metrics["test_weighted_f1"]

print("=" * 65)
print("PROJECT READINESS ASSESSMENT")
print("=" * 65)

print(f"Accuracy: {accuracy:.2%}")
print(f"Macro F1: {macro_f1:.2%}")
print(f"Weighted F1: {weighted_f1:.2%}")
print()

if accuracy >= 0.90 and macro_f1 >= 0.88:
    status = "STRONG"

    recommendation = (
        "The model is strong enough for the portfolio "
        "prototype. Continue to Phase 2 while retaining "
        "an uncertainty fallback."
    )

elif accuracy >= 0.82 and macro_f1 >= 0.80:
    status = "ACCEPTABLE"

    recommendation = (
        "The model is acceptable for a portfolio "
        "prototype. Continue to Phase 2 and route "
        "low-confidence predictions through a fallback."
    )

elif accuracy >= 0.75 and macro_f1 >= 0.72:
    status = "BORDERLINE"

    recommendation = (
        "You may develop Phase 2 in parallel, but the "
        "classifier should be improved before final "
        "integration."
    )

else:
    status = "INSUFFICIENT"

    recommendation = (
        "Do not rely on this model as the final routing "
        "layer yet. Consider more training or "
        "hyperparameter tuning."
    )

print("Status:", status)
print("Recommendation:", recommendation)

PROJECT READINESS ASSESSMENT
Accuracy: 87.74%
Macro F1: 87.22%
Weighted F1: 87.23%

Status: ACCEPTABLE
Recommendation: The model is acceptable for a portfolio prototype. Continue to Phase 2 and route low-confidence predictions through a fallback.


Cell 20 — Create a confidence-aware classifier function

In [20]:
def classify_intent(
    text: str,
    confidence_threshold: float = 0.70,
):
    result = classifier(text)[0]

    intent = result["label"]
    confidence = float(result["score"])

    return {
        "intent": intent,
        "confidence": confidence,
        "is_uncertain": confidence < confidence_threshold,
    }

example = classify_intent(
    "My card was stolen in London."
)

print(example)

{'intent': 'lost_or_stolen_card', 'confidence': 0.5324056148529053, 'is_uncertain': True}


In [21]:
focus_intents = [
    "virtual_card_not_working",
    "getting_virtual_card",
    "get_disposable_virtual_card",
    "why_verify_identity",
    "verify_my_identity",
    "card_swallowed",
    "declined_cash_withdrawal",
]

for intent in focus_intents:
    print("\n" + "=" * 90)
    print(intent)
    print("=" * 90)

    examples = dataset["train"].filter(
        lambda row: row["label_text"] == intent
    )

    for example in examples.select(
        range(min(15, len(examples)))
    ):
        print("-", example["text"])


virtual_card_not_working


Filter:   0%|          | 0/8993 [00:00<?, ? examples/s]

- Why is my disposable card not working?
- I can't get my virtual card to work.
- I can't get my virtual card to work at all
- What do I do if my disposable virtual card doesn't work?
- I tried to use my disposable virtual card to pay a subscription to the gym and it got rejected. Any ideas why?
- What do I have to do to get the virtual card to work?
- I received my Virtual card information, but was unable to use it to make a purchase. Why did this happen and what can I do?
- My throwaway virtual card won't work
- I cannot get my virtual card to function.
- I have a disposable card but it does not work?
- Why won't my virtual card work?
- My disposable virtual card isn't working.
- Why was my virtual card rejected?
- The virtual card won't work.
- This disposable virtual card is not working.

getting_virtual_card


Filter:   0%|          | 0/8993 [00:00<?, ? examples/s]

- I heard you have virtual cards. How do I get one?
- how to get virtual card
- where can i have a virtual card
- Where is my virtual card located?
- I want one of the virtual cards!
- I don't have a virtual card - how do I get one?
- Is there an alternative to a physical card?
- explain the virtual card
- I haven't received my virtual card yet!!
- I have lost my card, but need to place an online order! How do I get a virtual card instantly?
- How do I go about getting a virtual card?
- I did not get my virtual card yet, Why?
- I ordered a virtual card but it hasn't come through yet
- I want to get a virtual card!
- Is there a way to get a virtual card?

get_disposable_virtual_card


...

- What do you demand for identity verification?
- Tell me what the steps for the identity checks are
- I'm not sure what I need to verify my identity.
- How can I prove who I am?
- If I'm getting my identity verified, what all do I need?
- How can I prove I am me?
- Do I need any kind of proof for the identity check?
- Hi! What documents can I use to verify my identity?
- What are all of the different steps for identity checks?
- Where can I verify my identity?
- Where in the app do I go to verify my identity?
- What proof of identification is needed?
- I have to verify my identity. What do I need to do?
- How can I verify my identity?
- im worried about fraud how do you protect my account

card_swallowed


Filter:   0%|          | 0/8993 [00:00<?, ? examples/s]

- Why did the ATM swallow my card?
- how can the money machine keep my card what do i need to do?
- The ATM stole my card!
- I can't get my card out of the ATM
- The ATM kept my card?
- The ATM at Metro bank on High St. Kensington didn't return my card, and the bank is now closed. How do I get back a card swallowed by an ATM?
- My card is stuck inside the ATM, what am I supposed to do?
- Please send a new card; the ATM ate mine.
- What should I do with my stuck ATM?
- What are the steps to get my card back that was kept by the ATM?
- what do I need to do if the ATM kept my card?
- Who do I talk to about the ATM swallowing my card?
- An ATM machine didn't give me back my card.
- Will you please help me get my card back?
- Please help, the atm swallowed my card, what do I do?

declined_cash_withdrawal


Filter:   0%|          | 0/8993 [00:00<?, ? examples/s]

- Is there a problem with my account? When I tried to withdraw cash at an ATM I was denied.
- Please double check my card - withdrawal was working fine so far, but this morning on the way to work it suddenly got declined!
- what is my monthly spending limit because i was refused my money at an atm
- The ATM keeps declining my card! I tried two different ATMs already can you please check if everything is alright with my account??
- The ATM machines keep declining my card and I don't know why. I thought I had money in my account. Why is this happening?
- Would you please check my Card. As Withdrawal was working fine so far, but this morning suddenly got declined. Can you please check the problem?
- Why is my card being declined at the ATM? I have tried multiple ATMs and i keep running into the same problem. Could you verify that everything is okay with my account?
- I was denied cash at an ATM
- I am unable to get cash from the ATM
- Hi, In morning, i was trying to withdraw money from my